# 冻结模型高现金的输入机制
固定锚点来自用户关注的失败区间；同时保留所有2023+日期与四个事前状态。两种基准解释现金log-odds，不用未来收益排名特征，不重新拟合。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.model.minute_pit_allocator import _scaled
from src.workflow.pit_constituent_auxiliary_training_allocation import PitAuxiliaryAllocationNet,_project_legal_weights
from src.analysis.frozen_input_attribution import integrated_inputs
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';torch.set_num_threads(1)
card=root/'research/frozen_cash_input_explanation_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,plan['source'],['inputs/features','inputs/common','inputs/frozen_parent_models','inputs/plan'])
predicted,pred_audit=_source(root,plan['prediction_source'],['inputs/raw_predictions','inputs/market'])
state=source['inputs/frozen_parent_models']['outer_refit'];spec=source['inputs/plan']
features=source['inputs/features'];common=source['inputs/common'];dates=pd.DatetimeIndex(predicted['inputs/market'].signal_date)
dates=dates[(dates>=pd.Timestamp(plan['start']))&(dates<=pd.Timestamp(plan['end']))]
primary=features.reindex(dates)[state['primary_features']];aux=common.reindex(dates)[state['common_features']]
assert primary.notna().all().all() and aux.notna().all().all()
x=_scaled(primary,state['primary_features'],state['primary_scaler']);z=_scaled(aux,state['common_features'],state['common_scaler']);u=torch.cat([x,z],dim=1).double();n=x.shape[1]
net=PitAuxiliaryAllocationNet(primary_feature_count=n,common_feature_count=z.shape[1],hidden_units=spec['hidden_units'],tail_hidden_units=spec['tail_hidden_units']);net.load_state_dict({k:torch.tensor(v) for k,v in state['model_state'].items()});net=net.double().eval()
for p in net.parameters():p.requires_grad_(False)
def score(value):
 logits,_,_=net(value[:,:n],value[:,n:]);logits=logits.clamp(-8,8)
 return logits[:,2]-torch.logsumexp(logits[:,:2],dim=1)
with torch.no_grad():
 logits,_,_=net(u[:,:n],u[:,n:]);cash=_project_legal_weights(logits,maximum_soxl_weight=spec['maximum_soxl_weight'],minimum_cash_weight=spec['minimum_cash_weight'])[:,2].numpy()
expected=predicted['inputs/raw_predictions']['frozen_current'].reindex(dates).cash_weight.to_numpy();np.testing.assert_allclose(cash,expected,rtol=2e-6,atol=2e-7)
columns=['primary:'+c for c in state['primary_features']]+['common:'+c for c in state['common_features']]
# Same fixed baselines for every date; historical observations are never selected by outcomes.
old_x=_scaled(features[state['primary_features']].shift(5).reindex(dates),state['primary_features'],state['primary_scaler'])
old_z=_scaled(common[state['common_features']].shift(5).reindex(dates),state['common_features'],state['common_scaler'])
assert features[state['primary_features']].shift(5).reindex(dates).notna().all().all()
baselines={'training_mean':torch.zeros_like(u),'five_sessions_earlier':torch.cat([old_x,old_z],dim=1).double()}
all_attr={};errors={}
for name,baseline in baselines.items():
 values,residual=integrated_inputs(score,u,baseline,nodes=64);assert residual.abs().max()<1e-5
 all_attr[name]=pd.DataFrame(values.numpy(),index=dates,columns=columns);errors[name]=float(residual.abs().max())
def family(name):
 branch,key=name.split(':',1)
 if branch=='common':return 'common_tail_encoder'
 if 'breadth' in key or 'cross' in key:return 'cross_asset_breadth'
 if 'tracking' in key or 'levered_decay' in key:return 'tracking_and_decay'
 if any(k in key for k in ['vix','vol','rv','drawdown','dist_high','range_pos','panic','chop']):return 'volatility_drawdown_stress'
 return 'price_trend'
groups=pd.Series({c:family(c) for c in columns});r5=primary.soxx_ret5;r20=primary.soxx_ret20
regime=pd.Series(np.select([r5.ge(0)&r20.ge(0),r5.ge(0)&r20.lt(0),r5.lt(0)&r20.ge(0)],['uptrend','rebound','pullback'],default='decline'),index=dates)
aggregates=[];anchor=[]
for name,attrib in all_attr.items():
 grouped=attrib.T.groupby(groups).sum().T;grouped['year']=dates.year;grouped['regime']=regime;grouped['cash']=cash
 for (year,kind),part in grouped.groupby(['year','regime']):
  aggregates.append({'baseline':name,'year':year,'regime':kind,'sessions':len(part),'mean_cash':part.cash.mean(),**part.drop(columns=['year','regime','cash']).mean().to_dict()})
 for col in columns:
  anchor.append({'baseline':name,'feature':col,'family':family(col),'raw_value':float(pd.concat([primary,aux],axis=1).iloc[dates.get_loc(pd.Timestamp(plan['anchor'])),columns.index(col)]),'standardized_value':float(u[dates.get_loc(pd.Timestamp(plan['anchor'])),columns.index(col)]),'cash_log_odds_contribution':float(attrib.loc[plan['anchor'],col])})
anchor=pd.DataFrame(anchor);aggregates=pd.DataFrame(aggregates)
summary={'experiment':card.name,'decision':'frozen_model_input_diagnosis_no_feature_selection','new_fits':0,'new_backtests':0,'parameter_searches':0,'objective_achieved':False,'sessions':len(dates),'anchor':plan['anchor'],'anchor_raw_cash':float(cash[dates.get_loc(pd.Timestamp(plan['anchor']))]),'prediction_max_absolute_error':float(np.max(np.abs(cash-expected))),'integration_residual':errors,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Frozen pre-projection cash log-odds attribution; correlated inputs and interpolation are not causal or tradable counterfactuals. All years/regimes retained, no future return screening.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/attributions','analysis/scaled_inputs','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='frozen_feature_diagnosis',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,quadrature_nodes=64)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,prediction_audit=pred_audit,groups=groups)
 rec.save_objects(artifact_path='analysis',attributions=all_attr,scaled_inputs=pd.DataFrame(u.numpy(),index=dates,columns=columns),anchor=anchor,regime_year=aggregates,summary=summary)
 rec.save_objects(artifact_path='sources',attribution=(root/'src/analysis/frozen_input_attribution.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),maximum_integration_error=max(errors.values()))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
anchor.to_csv(card/'anchor_inputs.csv',index=False);aggregates.to_csv(card/'regime_year_attribution.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print(anchor.loc[anchor.baseline.eq('five_sessions_earlier')].sort_values('cash_log_odds_contribution',ascending=False).head(12).to_string(index=False))


In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.allocation_feature_contract import canonical_return_inputs
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/frozen_cash_input_explanation_20261014';plan=json.loads((card/'plan.json').read_text());summary=json.loads((card/'summary.json').read_text());qlib.init(provider_uri=summary['provider_uri'],region='us')
source,_=_source(root,plan['source'],['inputs/features','inputs/common','inputs/frozen_parent_models']);state=source['inputs/frozen_parent_models']['outer_refit']
primary=source['inputs/features'][state['primary_features']];common=source['inputs/common'][state['common_features']].reindex(primary.index)
canonical,aliases=canonical_return_inputs(primary,common)
rows=[]
for p in primary:
 for c in common:
  pair=pd.concat([primary[p].rename('p'),common[c].rename('c')],axis=1).loc[:'2022-12-30'].dropna()
  if len(pair)<100:continue
  rows.append({'primary':p,'common':c,'training_rows':len(pair),'correlation':pair.p.corr(pair.c),'maximum_raw_difference':float((pair.p-pair.c).abs().max())})
redundancy=pd.DataFrame(rows)
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
summary['research_contract']={'original_branch_fields':len(primary.columns)+len(common.columns),'canonical_fields':len(canonical.columns),'alias_pairs':len(aliases),'verified_rows':len(canonical),'frozen_model_inputs_changed':False,'production_release_unchanged':True}
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name)
 rec.save_objects(artifact_path='analysis',canonical_research_inputs=canonical,aliases=aliases,training_redundancy=redundancy,summary=summary)
 rec.save_objects(artifact_path='sources',feature_contract=(root/'src/data/allocation_feature_contract.py').read_text())
 artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');aliases.to_csv(card/'feature_aliases.csv',index=False)
print(summary['research_contract'])


# 冻结模型高现金：信息已存在，问题在组合响应与重复表示

本轮0拟合、0回测、0参数搜索，不依据未来收益排序或删除特征。原f696冻结网络与保存预测944日对齐，最大现金误差小于2e-6；生产release未变。

2023-11-03原始现金目标75.1029%。输入已包括SOXL三日涨幅23.3557%、公共分支SOXX五日涨幅6.9610%、VIX五日变化-29.9013%，故“没有看到反弹”并不成立。

采用训练标准化均值及五个交易日前真实向量两种固定参照，对投影前现金log-odds做64点积分梯度。两种归因的加总残差最大约3e-13，数值可核对。该归因不是现金百分点，不是投资收益；插值路径可能脱离真实联合分布，相关特征间贡献分配依赖参照，不是因果证据。

相对五日前，价格趋势组贡献+1.6680，公共编码分支+0.5255，风险/波动/回撤组-1.0808。soxx_trend60单项+1.6214，但相对训练均值的主导项变成semis_vs_cross_60（+1.7647）。不能只挑一种参照宣称找到唯一罪因。

在事前定义的修复状态（SOXX过去5日正、20日负）中，2023/2024/2025/2026样本数49/44/34/26；相对五日前，价格趋势组平均对现金log-odds的贡献依次+0.7205/+0.6524/+0.7558/+0.3691。说明该方向不是仅锚点一天出现，但这是同一冻结模型的描述，不等于四次独立预测验证，且不证明提高现金不合理。

训练截止前3020行的无标签冗余检查确认：primary soxx_ret5/20/60 与common mom5/20/60相关约1，最大原始差异约5.96e-8。主分支及公共分支重复表达同一收益概念；其余高相关波动/回撤项并不完全同义，未删。重复可以影响表示和解释，但不能由相关性直接推出过拟合或亏损因果。

已实现研究用canonical_return_inputs合同：将这三对同义字段统一，保持缺失标记，数值/日历不一致时报错。原55分支字段形成52个研究字段；两分支原网络、权重和标准化不改，去重视图不直接喂给冻结模型。用所有可比行验证别名，完整研究视图及相关矩阵仅存Recorder。

解决边界：完成了输入响应定位和同义特征合同，未证明删去重复信息能提升收益，也未修改仓位规则。下一步应对照实际信息变化解释修复过程，区分短期变化、长期受损程度、相对强弱及风险变化；先验证同名不同义、重复概念和跨时期语义一致性。不得把本次归因自动变为负号翻转、单调规则或新模型训练。

[锚点55项输入与两种归因](anchor_inputs.csv) · [所有年份与四状态](regime_year_attribution.csv) · [同义字段合同](feature_aliases.csv)

Recorder `8f591f545f7d4f8189411abafb84b542`。
